In [1]:
import os
import json
from dotenv import load_dotenv
from azure.identity import (DefaultAzureCredential, get_bearer_token_provider)
from azure.storage.blob import BlobServiceClient
from langchain_openai import (ChatOpenAI, OpenAIEmbeddings)
from langgraph.graph import (StateGraph, START, END)
from langchain.agents import create_agent

In [2]:
load_dotenv()

True

In [3]:
azure_project_endpoint = os.getenv("AZURE_PROJECT_ENDPOINT")
azure_model_endpoint = os.getenv("AZURE_MODEL_ENDPOINT")
azure_model_name = os.getenv("AZURE_MODEL_NAME")
azure_embedding_model_name = os.getenv("AZURE_EMBEDDING_MODEL_NAME")

In [4]:
credential = DefaultAzureCredential()

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

In [5]:
llm = ChatOpenAI(
    model=azure_model_name,
    base_url=f"{azure_project_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
    use_responses_api=True,
)

In [6]:
def load_employee_data():
    """Download employee JSON data from Azure Blob Storage."""
    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_EMPLOYEES"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_EMPLOYEES"]

    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"employees": [...]}.
        if isinstance(data, dict):
            data = data.get("employees")

        if not isinstance(data, list):
            raise ValueError(
                "Employee data must be a JSON list or an "
                "object containing an 'employees' list."
            )

        return data

    finally:
        credential.close()

In [7]:
def load_task_data():
    """Download task JSON data from Azure Blob Storage."""

    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_TASKS"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_TASKS"]
    
    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"tasks": [...]}.
        if isinstance(data, dict):
            data = data.get("tasks")

        if not isinstance(data, list):
            raise ValueError(
                "Task data must be a JSON list or an "
                "object containing a 'tasks' list."
            )

        return data

    finally:
        credential.close()

In [8]:
employee_data = load_employee_data()
print('employee_data ', employee_data)

employee_data  [{'id': 0, 'name': 'Jonas Greslund', 'position': 'Software Engineer', 'skill_map': {'angular': 'expert', 'dotnet': 'advanced', 'python': 'advanced', 'sql': 'advanced', 'pulumi': 'novice'}}, {'id': 1, 'name': 'Link', 'position': 'Software Engineer', 'skill_map': {'angular': 'advanced', 'dotnet': 'intermediate', 'python': 'advanced', 'sql': 'intermediate', 'pulumi': 'intermediate'}}, {'id': 2, 'name': 'Zeenula Pelphegopoulos', 'position': 'Senior Software Engineer', 'skill_map': {'angular': 'intermediate', 'dotnet': 'advanced', 'python': 'expert', 'sql': 'advanced', 'pulumi': 'expert'}}, {'id': 3, 'name': 'Peach', 'position': 'Software Quality Engineer', 'skill_map': {'selenium': 'advanced', 'regression_testing': 'advanced'}}]


In [9]:
task_data = load_task_data()
print('task_data ', task_data)

task_data  [{'id': 0, 'title': 'Handle the edge case with the UI of the scoring performance chart.', 'description': 'There are times when 2 different z-scores have a value of zero.', 'required_skills': ['Angular', 'css']}, {'id': 1, 'title': 'Deploy infrastructure', 'description': 'Provision cloud resources using Pulumi.', 'required_skills': ['pulumi', 'python']}, {'id': 2, 'title': 'Optimize database queries', 'description': 'Improve the performance of database queries to reduce latency.', 'required_skills': ['sql', 'python']}, {'id': 3, 'title': 'Conduct regression testing', 'description': 'Perform regression testing to ensure that new code changes do not break existing functionality.', 'required_skills': ['selenium', 'regression_testing']}]


In [10]:
def evaluate_employee_skill_set(employee_name, skill, level, evalation_agent):
    """
    This function returns an evaluation of a particular employee's skills.
    """

    evaluation_prompt = f"""
        The employee's name is {employee_name}.
        The skill under evaluation is {skill}.
        The level of expertise is {level}.
    """

    result = evaluation_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    })
    return result

In [11]:
system_prompt = (
    "You are an employee skill evaluator. "
    "Your job is to read an employee's skill as well as how advanced they are at that particular skill. "
    "You must output a brief statement about their level at that skill."
)

evaluation_agent = create_agent(
    model=llm,
    tools=[], # We can add tools later.
    system_prompt=system_prompt
)

In [12]:
evaluations = []
for employee in employee_data:
    for skill, level in employee['skill_map'].items():
        evaluation = evaluate_employee_skill_set(employee['name'], skill, level, evaluation_agent)
        evaluations.append(evaluation)

In [13]:
# Initialize with an embedding model
embeddings_model = OpenAIEmbeddings(
    model=azure_embedding_model_name,
    base_url=f"{azure_model_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)

In [14]:
evaluation_content = []
for evaluation in evaluations:
    content = evaluation['messages'][1].content
    evaluation_content.append(content[0]['text'])
embeddings = embeddings_model.embed_documents(["one"])
embeddings

[[-0.00769805908203125,
  -0.005245208740234375,
  0.00673675537109375,
  -0.0229339599609375,
  0.0008225440979003906,
  0.01116943359375,
  0.021514892578125,
  0.08428955078125,
  0.007259368896484375,
  0.02655029296875,
  -0.007221221923828125,
  -0.00943756103515625,
  -0.0166778564453125,
  -0.035400390625,
  -0.002826690673828125,
  -0.004856109619140625,
  -0.052459716796875,
  -0.0022106170654296875,
  -0.005908966064453125,
  -0.008544921875,
  0.006572723388671875,
  -0.01200103759765625,
  -0.00865936279296875,
  0.00188446044921875,
  -0.00777435302734375,
  0.0143280029296875,
  0.034820556640625,
  -0.0156402587890625,
  -0.0216217041015625,
  0.01261138916015625,
  0.013031005859375,
  0.03851318359375,
  -0.0001621246337890625,
  -0.00445556640625,
  -0.01345062255859375,
  -0.00119781494140625,
  0.03179931640625,
  -0.0157012939453125,
  -0.01270294189453125,
  0.0040283203125,
  0.007568359375,
  0.01947021484375,
  -0.03717041015625,
  0.0180206298828125,
  0.0330

In [15]:
task_data

[{'id': 0,
  'title': 'Handle the edge case with the UI of the scoring performance chart.',
  'description': 'There are times when 2 different z-scores have a value of zero.',
  'required_skills': ['Angular', 'css']},
 {'id': 1,
  'title': 'Deploy infrastructure',
  'description': 'Provision cloud resources using Pulumi.',
  'required_skills': ['pulumi', 'python']},
 {'id': 2,
  'title': 'Optimize database queries',
  'description': 'Improve the performance of database queries to reduce latency.',
  'required_skills': ['sql', 'python']},
 {'id': 3,
  'title': 'Conduct regression testing',
  'description': 'Perform regression testing to ensure that new code changes do not break existing functionality.',
  'required_skills': ['selenium', 'regression_testing']}]

In [16]:
evaluation_content

['Jonas Greslund demonstrates expert-level proficiency in Angular, with advanced knowledge of the framework and the ability to build and maintain complex applications.',
 'Jonas Greslund demonstrates advanced proficiency in .NET.',
 'Jonas Greslund demonstrates advanced proficiency in Python, with the ability to tackle complex programming tasks and develop robust, maintainable solutions.',
 'Jonas Greslund demonstrates advanced SQL proficiency, with strong capabilities in complex querying, data manipulation, and query optimization.',
 'Jonas Greslund is a novice in Pulumi, with a foundational understanding and a need for guidance when defining and managing infrastructure as code.',
 'Link demonstrates advanced proficiency in Angular, with strong expertise in developing and maintaining complex applications using the framework.',
 'Link demonstrates intermediate proficiency in .NET, with a solid grasp of core concepts and the ability to handle common development tasks independently.',
 '